# 🏭 ETL clásico


Extract → Clean → Join
→ Transform → Load

## 1️⃣ EXTRACT

In [1]:
from pyspark.sql import (
    SparkSession, functions as F)

spark = (
    SparkSession.builder
    .appName('etl_clasico')
    .getOrCreate()
)

In [2]:
# Ventas crudas (con basura típica)
ventas = spark.createDataFrame([
  ('1','P001','2025-01-05','120€'),
  ('2','P002','2025-01-05','80€'),
  ('2','P002','2025-01-05','80€'),
  ('3','P003','2025-02-10','300€'),
  ('4','P001','2025-02-11', None),
  ('5','P002','2025-03-01','50€'),
], ['id','producto','fecha','importe'])

# Catálogo de productos
productos = spark.createDataFrame([
  ('P001','Camiseta','Ropa'),
  ('P002','Café','Alimentación'),
  ('P003','Portátil','Electrónica'),
], ['producto','nombre','categoria'])

ventas.show()

+---+--------+----------+-------+
| id|producto|     fecha|importe|
+---+--------+----------+-------+
|  1|    P001|2025-01-05|   120€|
|  2|    P002|2025-01-05|    80€|
|  2|    P002|2025-01-05|    80€|
|  3|    P003|2025-02-10|   300€|
|  4|    P001|2025-02-11|   NULL|
|  5|    P002|2025-03-01|    50€|
+---+--------+----------+-------+



## 2️⃣ CLEAN

In [3]:
ventas_limpias = (
  ventas
  # 1. Importe: '120€' -> 120 (entero).
  #    Nulos y basura -> null
  .withColumn('importe',
    F.regexp_replace('importe', '€', '')
     .cast('int'))
  # 2. Fecha texto -> tipo date
  .withColumn('fecha',
    F.to_date('fecha'))
  # 3. Fuera nulos Y casts fallidos
  .filter(F.col('importe').isNotNull())
  # 4. Dedup al final: shuffle sobre
  #    menos filas
  .dropDuplicates()
)

ventas_limpias.show()

+---+--------+----------+-------+
| id|producto|     fecha|importe|
+---+--------+----------+-------+
|  1|    P001|2025-01-05|    120|
|  2|    P002|2025-01-05|     80|
|  3|    P003|2025-02-10|    300|
|  5|    P002|2025-03-01|     50|
+---+--------+----------+-------+



## 3️⃣ JOIN
### Enriquecemos las ventas

Añadimos nombre y categoría.

In [4]:
ventas_enriq = (
  ventas_limpias
  .join(
    F.broadcast(productos),
    on='producto',
    how='left')
)

ventas_enriq.show()

+--------+---+----------+-------+--------+------------+
|producto| id|     fecha|importe|  nombre|   categoria|
+--------+---+----------+-------+--------+------------+
|    P001|  1|2025-01-05|    120|Camiseta|        Ropa|
|    P002|  2|2025-01-05|     80|    Café|Alimentación|
|    P003|  3|2025-02-10|    300|Portátil| Electrónica|
|    P002|  5|2025-03-01|     50|    Café|Alimentación|
+--------+---+----------+-------+--------+------------+



## 4️⃣ TRANSFORM
### Creamos la métrica

Columna 'mes' y el total
facturado por categoría.

In [5]:
ventas_enriq = ventas_enriq.withColumn(
  'mes', F.month('fecha'))

resumen = (
  ventas_enriq
  .groupBy('categoria')
  .agg(
    F.sum('importe').alias('total'),
    F.count('*').alias('n_ventas'))
  .orderBy(F.col('total').desc())
)

resumen.show()

+------------+-----+--------+
|   categoria|total|n_ventas|
+------------+-----+--------+
| Electrónica|  300|       1|
|Alimentación|  130|       2|
|        Ropa|  120|       1|
+------------+-----+--------+



## 5️⃣ LOAD
### Guardamos el resultado

In [6]:
ruta = '/tmp/ventas_procesadas'

(ventas_enriq
 .write
 .mode('overwrite')
 .partitionBy('categoria')
 .parquet(ruta))

# Comprobamos leyendo de vuelta
leido = spark.read.parquet(ruta)
print(f"Guardadas {leido.count()} filas")
leido.show()

Guardadas 4 filas
+--------+---+----------+-------+--------+---+------------+
|producto| id|     fecha|importe|  nombre|mes|   categoria|
+--------+---+----------+-------+--------+---+------------+
|    P002|  2|2025-01-05|     80|    Café|  1|Alimentación|
|    P002|  5|2025-03-01|     50|    Café|  3|Alimentación|
|    P003|  3|2025-02-10|    300|Portátil|  2| Electrónica|
|    P001|  1|2025-01-05|    120|Camiseta|  1|        Ropa|
+--------+---+----------+-------+--------+---+------------+



In [7]:
spark.stop()